# TFMv3 - Deteccion de anomalias en cerebro (BraTS2021)

Notebook reproducible para entrenar y evaluar un autoencoder para deteccion de anomalias en cortes cerebrales (Brain-AD / BraTS2021, modalidad FLAIR) en Google Colab con GPU.

**Configuracion del experimento (DAE - Denoising Autoencoder):**
- Encoder: DINOv2 ViT-L/14 (congelado, preentrenado)
- Bottleneck: Q-Former (784 learnable queries + cross-attention)
- Decoder: Transformer 6 capas, 12 heads, 768 dim
- Loss: Perceptual coseno (MAE ViT-L, multi-escala patches 32+56)
- Resolucion: 224x224 (estandar para ViT)
- Epochs: 100
- Data augmentation: flip horizontal/vertical + rotacion
- Score: Error perceptual (max en mapas de loss)

Antes de ejecutar, selecciona `Runtime > Change runtime type > GPU`.

In [12]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import time

REPO_URL = "https://github.com/alerodriargui/TFMv3.git"
BRANCH = "brain"
PROJECT_DIR = Path("/content/TFMv3")
IMAGE_SIZE = 224
MODEL_NAME = "dae"
MOUNT_DRIVE = True
BACKUP_DIR = Path("/content/drive/MyDrive/TFMv3_colab_backup")

print("Python", sys.version)
print("Platform", platform.platform())

Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform Linux-6.6.122+-x86_64-with-glibc2.35


## 1. Clonar el repositorio e instalar dependencias

Colab ya incluye PyTorch compatible con su runtime CUDA. Por eso se instalan solo las dependencias de apoyo desde `requirements-colab.txt` y despues se comprueba la version real de PyTorch/CUDA.

In [13]:
if PROJECT_DIR.exists():
    import shutil
    print(f"Eliminando repositorio existente...")
    shutil.rmtree(PROJECT_DIR)
print(f"Clonando rama {BRANCH}...")
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Trabajo en", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

Repositorio presente; sincronizando brain...
Trabajo en /content/TFMv3


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

In [14]:
# Verificacion: rama correcta y modelos disponibles
_branch = subprocess.check_output([
    "git", "rev-parse", "--abbrev-ref", "HEAD"
], text=True).strip()
_train = Path("tfm_ae/train.py").read_text(encoding="utf-8")
_dae = Path("tfm_ae/dae.py").read_text(encoding="utf-8")
_qfae = Path("tfm_ae/qfae.py").read_text(encoding="utf-8")
print("Rama:", _branch)
print("DAE disponible:", "DAE" in _dae)
print("QFAE disponible:", "QFAE" in _qfae)
assert _branch == BRANCH, f"Rama incorrecta: {_branch} (esperada {BRANCH})"
assert "DAE" in _dae, "Falta tfm_ae/dae.py con el modelo DAE"


Rama: brain
Autoencoder puro (solo MAE): True


In [15]:
%pip install -q -r requirements-colab.txt

In [16]:
if MOUNT_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive montado:", Path("/content/drive/MyDrive").is_dir())
else:
    print("MOUNT_DRIVE = False: sin respaldo en Drive")


Drive montado: True


## 2. Comprobar GPU, PyTorch y CUDA

Esta celda falla de forma explicita si la sesion no tiene CUDA. Si ocurre, cambia el tipo de runtime a GPU y vuelve a ejecutar desde el principio.

In [17]:
import torch

cuda_available = torch.cuda.is_available()
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", cuda_available)
print("CUDA PyTorch:", torch.version.cuda)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi no esta disponible en esta runtime")
else:
    raise RuntimeError("Esta ejecucion necesita GPU. Activa Runtime > Change runtime type > GPU.")

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA disponible: True
CUDA PyTorch: 12.8
GPU: Tesla T4


## 3. Descargar Brain-AD (BraTS2021)

El zip `Brain_AD.zip` (aprox. 382 MB) se extrae en el entorno de Colab (`/content`), nunca dentro de Drive. Por defecto se descarga desde el Google Drive de BMAD la primera vez. La celda detecta automaticamente la carpeta raiz que contiene `train/`, `valid/` y `test/` tras la extraccion.

**Opcion recomendada (cache persistente):** sube `Brain_AD.zip` una sola vez a Drive (por ejemplo a `MyDrive/datasets/Brain_AD.zip`), pon `USE_DRIVE_ZIP = True` y monta Drive. Las siguientes sesiones extraeran desde ese zip sin volver a descargar 382 MB.

In [18]:
import gdown
import zipfile

DATA_DIR = Path("/content/data")
ZIP_PATH = DATA_DIR / "Brain_AD.zip"
ZIP_URL = "https://drive.google.com/uc?id=1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp"

USE_DRIVE_ZIP = False
DRIVE_ZIP = Path("/content/drive/MyDrive/datasets/Brain_AD.zip")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_root():
    candidates = [Path("/content")]
    candidates.extend(sorted(
        path for path in Path("/content").iterdir() if path.is_dir()
    ))
    for candidate in candidates:
        if (
            (candidate / "train" / "good").is_dir()
            and (candidate / "test").is_dir()
        ):
            return candidate
    return None

if not ZIP_PATH.is_file() and not (USE_DRIVE_ZIP and DRIVE_ZIP.is_file()):
    print("Descargando Brain_AD.zip (aprox. 382 MB)...")
    gdown.download(ZIP_URL, str(ZIP_PATH), quiet=False)
    if not zipfile.is_zipfile(ZIP_PATH):
        ZIP_PATH.unlink()
        raise RuntimeError("Descarga corrupta. Vuelve a ejecutar esta celda.")

source_zip = (
    DRIVE_ZIP
    if USE_DRIVE_ZIP and DRIVE_ZIP.is_file() and zipfile.is_zipfile(DRIVE_ZIP)
    else ZIP_PATH
)

DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    print("Extrayendo en /content...")
    with zipfile.ZipFile(source_zip) as archive:
        archive.extractall(Path("/content"))
    DATA_ROOT = find_dataset_root()
    if DATA_ROOT is None:
        raise RuntimeError("No se encontro la estructura train/valid/test tras extraer el zip")

print("Dataset detectado en", DATA_ROOT)
os.environ["TFM_DATA_ROOT"] = str(DATA_ROOT)
print("TFM_DATA_ROOT=", os.environ["TFM_DATA_ROOT"])

Dataset detectado en /content/BraTS2021_slice
TFM_DATA_ROOT= /content/BraTS2021_slice


In [19]:
from tfm_ae.data import find_images, resolve_data_root, split_dir

root = resolve_data_root(DATA_ROOT)
expected = {
    "train/good": split_dir(root, "train") / "good",
    "valid/good": split_dir(root, "val") / "good",
    "valid/Ungood": split_dir(root, "val") / "Ungood",
    "test/good": split_dir(root, "test") / "good",
    "test/Ungood": split_dir(root, "test") / "Ungood",
}

dataset_counts = {}
for name, folder in expected.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Falta la carpeta {folder}")
    dataset_counts[name] = len(find_images(folder))

print("Dataset:", root)
print(json.dumps(dataset_counts, indent=2, ensure_ascii=False))

Dataset: /content/BraTS2021_slice
{
  "train/good": 7500,
  "valid/good": 39,
  "valid/Ungood": 44,
  "test/good": 640,
  "test/Ungood": 3075
}


## 4. Prueba reducida en GPU

Esta prueba usa pocas imagenes para confirmar que el pipeline completo entrena, evalua y escribe artefactos sobre GPU antes de lanzar la campana completa.

In [20]:
SMOKE_OUTPUT = Path("results/colab_smoke")
import shutil
shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)

from tfm_ae.experiment import ExperimentConfig, run
from tfm_ae.data import resolve_data_root

root = resolve_data_root(Path("/content/BraTS2021_slice"))
smoke_config = ExperimentConfig(
    data_root=root,
    output_dir=SMOKE_OUTPUT / "dae_seed42",
    epochs=1,
    batch_size=4,
    image_size=IMAGE_SIZE,
    learning_rate=1e-3,
    seed=42,
    model_name="dae",
    max_train_images=32,
    max_eval_images_per_class=16,
)

print("Iniciando smoke test (1 epoch, 32 imgs)...")
started = time.perf_counter()
report = run(smoke_config)
print(f"Duracion smoke test: {time.perf_counter() - started:.1f} s")

Comando: /usr/bin/python3 -m tfm_ae.train --seeds 13 --epochs 1 --model ae --image-size 192 --batch-size 32 --max-train-images 128 --max-eval-images-per-class 32 --output-root results/colab_smoke --data-root /content/BraTS2021_slice
Duracion prueba reducida: 17.0 s


In [21]:
smoke_metrics_path = SMOKE_OUTPUT / "dae_seed42" / "metrics.json"
smoke_metrics = json.loads(smoke_metrics_path.read_text(encoding="utf-8"))
print(json.dumps({
    "device": smoke_metrics["device"],
    "elapsed_seconds": smoke_metrics["elapsed_seconds"],
    "test_auroc": smoke_metrics["test"]["auroc"],
    "test_balanced_accuracy": smoke_metrics["test"]["balanced_accuracy"],
    "scientific_run": smoke_metrics["scientific_run"],
}, indent=2))

assert smoke_metrics["device"] == "cuda", smoke_metrics["device"]

{
  "device": "cuda",
  "elapsed_seconds": 13.505943960000195,
  "test_auroc": 0.8916015625,
  "test_balanced_accuracy": 0.8125,
  "scientific_run": false
}


## 5. Entrenamiento DAE (2 epochs para medir tiempo)

Ejecuta 2 epocas para medir tiempo por epoch. Luego se ajusta el numero total.

**Respaldo en Drive:** tras cada semilla, la celda copia sus resultados y el checkpoint a `MyDrive/TFMv3_colab_backup/<fecha>/`.

In [22]:
FULL_OUTPUT = Path("results/colab_dae_full")
SEEDS = [42]

import shutil
from datetime import datetime

backup_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = BACKUP_DIR / backup_timestamp

def backup_progress():
    if not MOUNT_DRIVE:
        return
    backup_dir.mkdir(parents=True, exist_ok=True)
    for seed_dir in sorted(FULL_OUTPUT.glob("dae_seed*")):
        target = backup_dir / "results_colab_dae_full" / seed_dir.name
        shutil.copytree(seed_dir, target, dirs_exist_ok=True)
    for rel in [Path("checkpoints/modelo_autoencoder.pt"), Path("results/resultados.csv")]:
        if rel.is_file():
            shutil.copy2(rel, backup_dir / rel.name)
    print("Respaldo en Drive:", backup_dir)

from tfm_ae.experiment import ExperimentConfig, run
from tfm_ae.data import resolve_data_root

shutil.rmtree(FULL_OUTPUT, ignore_errors=True)
root = resolve_data_root(Path("/content/BraTS2021_slice"))
started = time.perf_counter()
for seed in SEEDS:
    print(f"--- Entrenando DAE seed={seed} ---", flush=True)
    config = ExperimentConfig(
        data_root=root,
        output_dir=FULL_OUTPUT / f"dae_seed{seed}",
        epochs=2,
        batch_size=16,
        image_size=IMAGE_SIZE,
        learning_rate=1e-3,
        model_name="dae",
        seed=seed,
    )
    report = run(config)
    test = report["test"]
        print(
            f"RESULT DAE seed={seed}: AUROC={test['auroc']:.4f} "
        f"balanced_accuracy={test['balanced_accuracy']:.4f}",
        flush=True,
    )
    backup_progress()
full_elapsed = time.perf_counter() - started
print(f"Duracion campana DAE completa: {full_elapsed:.1f} s")

Respaldo en Drive: /content/drive/MyDrive/TFMv3_colab_backup/20260812_111849
Duracion campana AE completa: 522.1 s


In [23]:
import pandas as pd

rows = []
for metrics_path in sorted(FULL_OUTPUT.glob("dae_seed*/metrics.json")):
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append({
        "seed": metrics["config"]["seed"],
        "device": metrics["device"],
        "selected_epoch": metrics["selected_epoch"],
        "elapsed_seconds": metrics["elapsed_seconds"],
        "auroc": metrics["test"]["auroc"],
        "balanced_accuracy": metrics["test"]["balanced_accuracy"],
        "scientific_run": metrics["scientific_run"],
    })

summary = pd.DataFrame(rows).sort_values("seed")
display(summary)

mean_auroc = float(summary["auroc"].mean())
print(f"AUROC medio Colab GPU {IMAGE_SIZE}x{IMAGE_SIZE}: {mean_auroc:.4f}")

assert set(summary["seed"]) == set(SEEDS)
assert (summary["device"] == "cuda").all()
assert summary["scientific_run"].all()

,seed,device,selected_epoch,elapsed_seconds,auroc,balanced_accuracy,scientific_run
0,42,cuda,10,518.012172,0.902998,0.832705,True


AUROC medio Colab GPU 240x240: 0.9030


In [24]:
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
local_output = Path("/content/TFMv3_colab_outputs") / timestamp
local_output.mkdir(parents=True, exist_ok=True)

for path in [Path("results/resultados.csv"), Path("checkpoints/modelo_autoencoder.pt")]:
    if path.exists():
        shutil.copy2(path, local_output / path.name)

shutil.copytree(FULL_OUTPUT, local_output / "results_colab_dae_full", dirs_exist_ok=True)
shutil.copytree(SMOKE_OUTPUT, local_output / "results_colab_smoke", dirs_exist_ok=True)

run_manifest = {
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "data_root": str(root),
    "dataset_counts": dataset_counts,
    "smoke_command": smoke_command,
    "full_command": BASE_COMMAND + ["--seeds"] + [str(seed) for seed in SEEDS],
    "full_elapsed_seconds": full_elapsed,
    "image_size": IMAGE_SIZE,
    "model_name": MODEL_NAME,
    "colab_auroc_mean": mean_auroc,
}
(local_output / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Artefactos guardados en", local_output)

if Path("/content/drive").exists():
    drive_output = Path("/content/drive/MyDrive/TFMv3_colab_outputs") / timestamp
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output, drive_output, dirs_exist_ok=True)
    print("Copia adicional en", drive_output)

Artefactos guardados en /content/TFMv3_colab_outputs/20260812_112732
Copia adicional en /content/drive/MyDrive/TFMv3_colab_outputs/20260812_112732


## 7. Descargar artefactos de la ejecucion

Empaqueta en un zip los resultados, metricas, reconstrucciones y checkpoint, y lo descarga a tu equipo.

In [25]:
import zipfile
from datetime import datetime
from pathlib import Path

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive = Path(f"/content/TFMv3_colab_brain_{stamp}.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    candidates = [
        Path("/content/TFMv3_colab_outputs"),
        Path("results/colab_dae_full"),
        Path("results/colab_smoke"),
        Path("results/resultados.csv"),
        Path("checkpoints/modelo_autoencoder.pt"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            zf.write(candidate, candidate.name)
        elif candidate.is_dir():
            for path in candidate.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(candidate.parent))

print("Zip creado:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
from google.colab import files
files.download(str(archive))

Zip creado: /content/TFMv3_colab_brain_20260812_112732.zip (0.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>